# KNOTS-05 — Dynamique des tresses : action d'Artin, croissance des longueurs, spectre de Perron

**Série** : SymbolicAI / Lean — Knots Theory (compagnon dynamique de 17a/17b/17c/17d)

KNOTS-02 calcule les invariants numériques d'un nœud, KNOTS-04 sa géométrie hyperbolique, KNOTS-03 interroge le lac `knot_lean`. Ce carnet-ci ajoute la couche que les précédents laissent de côté : **ce que fait une tresse quand on l'itère**.

Une tresse est un mot dans un groupe ; l'itérer, c'est appliquer encore et encore le même automorphisme. Ce carnet mesure trois choses et les confronte :

1. les **relations** que l'action d'Artin doit satisfaire — vérifiées par assertion, pas affirmées ;
2. la **croissance** de la longueur réduite d'un élément du groupe libre sous itération ;
3. le **spectre de Perron** de la matrice de transition associée, et l'écart avec la croissance mesurée.

### Ce que ce carnet ne prétend pas

- Il ne calcule pas la **dilatation** d'une classe d'application par train track (Bestvina–Handel) : c'est un chantier distinct, et le carnet dit à quel endroit il s'arrête.
- Il n'affirme **aucune égalité** entre croissance mesurée et rayon spectral : il mesure l'accord là où il tient, et montre les cas où il ne tient pas.
- L'**action d'Artin** sur le groupe libre n'est pas la représentation de Burau. Les deux existent ; ce carnet n'en utilise qu'une et ne les confond pas.

### Plan

| Section | Objet |
|---|---|
| 1 | Une tresse est un mot — et une permutation, qui ne suffit pas |
| 2 | L'action d'Artin sur le groupe libre, vérifiée par ses relations |
| 3 | Itérer : la croissance d'une longueur |
| 4 | Le spectre de Perron : une borne, et l'endroit où elle est atteinte |
| 5 | Le niveau abélianisé, qui ne voit rien |

In [1]:
import math
import sys

import numpy as np

print("Python", sys.version.split()[0])
print("numpy", np.__version__)

Python 3.13.15
numpy 2.4.6


## 1. Une tresse est un mot — et une permutation, qui ne suffit pas

Une tresse sur `n` brins s'écrit comme un mot en les générateurs `σ_i` et leurs inverses `σ_i^{-1}`, avec `1 <= i < n`. On la représente ici par une liste d'entiers : `+i` pour `σ_i`, `-i` pour `σ_i^{-1}`.

Le premier invariant gratuit est la **permutation** des brins : `σ_i` et `σ_i^{-1}` échangent les brins `i` et `i+1`, donc la permutation ne distingue pas un générateur de son inverse. C'est une information réelle, mais grossière — la suite du carnet vit précisément dans ce qu'elle ne voit pas.

In [2]:
def permutation(tresse, n):
    '''Permutation des brins lue par un mot de tresse (le signe ne compte pas).'''
    brins = list(range(1, n + 1))
    for g in tresse:
        i = abs(g)
        brins[i - 1], brins[i] = brins[i], brins[i - 1]
    return tuple(brins)


n = 3
gauche = [1, 2, 1]      # sigma1 sigma2 sigma1
droite = [2, 1, 2]      # sigma2 sigma1 sigma2

print("sigma1 sigma2 sigma1 ->", permutation(gauche, n))
print("sigma2 sigma1 sigma2 ->", permutation(droite, n))
assert permutation(gauche, n) == permutation(droite, n)

# Deux mots differents, meme permutation : la permutation ne les separe pas.
a = [1, 2]
b = [1, 2, 1, 1]
print("sigma1 sigma2        ->", permutation(a, n))
print("sigma1 sigma2 sigma1^2 ->", permutation(b, n))
assert permutation(a, n) == permutation(b, n)
print("les deux mots sont distincts :", a != b)

sigma1 sigma2 sigma1 -> (3, 2, 1)
sigma2 sigma1 sigma2 -> (3, 2, 1)
sigma1 sigma2        -> (2, 3, 1)
sigma1 sigma2 sigma1^2 -> (2, 3, 1)
les deux mots sont distincts : True


### Lecture du résultat

Les deux écritures de la relation de tresse donnent la **même** permutation : c'est une condition nécessaire, et l'assertion la contrôle. La dernière ligne du calcul montre le point qui compte : deux mots distincts (`σ1σ2` et `σ1σ2σ1²`) portent la même permutation. Une permutation est donc un **invariant trop faible** pour la question posée — il faut descendre au groupe, c'est-à-dire garder l'ordre des lettres et non seulement leur parité d'échange.

C'est exactement ce que fait la section suivante : remplacer la permutation des brins par un **automorphisme d'un groupe non abélien**.

## 2. L'action d'Artin sur le groupe libre, vérifiée par ses relations

Soit `F_n` le groupe libre engendré par `x_1, ..., x_n`. L'action d'Artin envoie le générateur `σ_i` sur l'automorphisme

```
σ_i(x_i)     = x_i · x_{i+1} · x_i^{-1}
σ_i(x_{i+1}) = x_i
σ_i(x_j)     = x_j            (j différent de i, i+1)
```

Un mot du groupe libre est ici un tuple d'entiers non nuls (`+k` pour `x_k`, `-k` pour `x_k^{-1}`), et la **réduction libre** supprime les paires `x_k x_k^{-1}` adjacentes.

Un automorphisme est écrit comme une fonction sur les lettres ; composer une liste `[a_1, a_2, ..., a_m]` signifie **appliquer `a_1` d'abord**, puis `a_2`, etc. — l'ordre de lecture du mot de tresse.

Rien de ce qui suit n'est supposé : les relations sont **assertées**, et une formule fausse ferait échouer la cellule.

In [3]:
def reduire(mot):
    '''Reduction libre d'un mot : supprime les paires x_k x_k^-1 adjacentes.'''
    sortie = []
    for g in mot:
        if sortie and sortie[-1] == -g:
            sortie.pop()
        else:
            sortie.append(g)
    return tuple(sortie)


def sigma(n, i, inverse=False):
    '''Automorphisme de F_n induit par sigma_i (ou son inverse).'''
    if not inverse:
        def f(mot):
            sortie = []
            for g in mot:
                if g == i:
                    sortie += [i, i + 1, -i]        # x_i -> x_i x_{i+1} x_i^-1
                elif g == -i:
                    sortie += [i, -(i + 1), -i]
                elif g == i + 1:
                    sortie.append(i)                # x_{i+1} -> x_i
                elif g == -(i + 1):
                    sortie.append(-i)
                else:
                    sortie.append(g)
            return sortie
    else:
        def f(mot):
            sortie = []
            for g in mot:
                if g == i:
                    sortie.append(i + 1)            # x_i -> x_{i+1}
                elif g == -i:
                    sortie.append(-(i + 1))
                elif g == i + 1:
                    sortie += [-(i + 1), i, i + 1]  # x_{i+1} -> x_{i+1}^-1 x_i x_{i+1}
                elif g == -(i + 1):
                    sortie += [-(i + 1), -i, i + 1]
                else:
                    sortie.append(g)
            return sortie
    return f


def appliquer(automorphismes, mot):
    '''Applique la liste dans l'ordre : le premier element agit d'abord.'''
    for aut in automorphismes:
        mot = reduire(tuple(x for lettre in mot for x in aut((lettre,))))
    return mot


def verifier_relations(n, verbose=True):
    '''Verifie sigma_i sigma_i^-1 = id, les relations d'Artin et les commutations.'''
    generateurs = [(k,) for k in range(1, n + 1)]
    controles = 0

    for i in range(1, n):
        for g in generateurs:
            assert appliquer([sigma(n, i), sigma(n, i, True)], g) == g
            controles += 1

    for i in range(1, n - 1):
        for g in generateurs:
            gauche = appliquer([sigma(n, i), sigma(n, i + 1), sigma(n, i)], g)
            droite = appliquer([sigma(n, i + 1), sigma(n, i), sigma(n, i + 1)], g)
            assert gauche == droite, (i, g, gauche, droite)
            controles += 1

    for i in range(1, n - 2):
        for g in generateurs:
            gauche = appliquer([sigma(n, i), sigma(n, i + 2)], g)
            droite = appliquer([sigma(n, i + 2), sigma(n, i)], g)
            assert gauche == droite, (i, g)
            controles += 1

    if verbose:
        print("n = {} : {} controles d'egalite de mots, tous verts".format(n, controles))
    return controles


total = sum(verifier_relations(n) for n in (2, 3, 4, 5))
print("total des controles :", total)

n = 2 : 2 controles d'egalite de mots, tous verts
n = 3 : 9 controles d'egalite de mots, tous verts
n = 4 : 24 controles d'egalite de mots, tous verts
n = 5 : 45 controles d'egalite de mots, tous verts
total des controles : 80


### Lecture du résultat

Les trois familles de relations tiennent **comme égalités de mots du groupe libre**, pour `n` de 2 à 5, sur chaque générateur : l'inverse, la relation de tresse `σ_i σ_{i+1} σ_i = σ_{i+1} σ_i σ_{i+1}`, et la commutation des générateurs distants `σ_i σ_{i+2} = σ_{i+2} σ_i`. Le nombre de contrôles affiché est celui qui a réellement été exécuté, pas un décompte recopié.

Ce point mérite d'être souligné : une implémentation fausse des formules d'Artin — une seule lettre mal placée dans l'image de `x_{i+1}` — fait **échouer** la cellule. La vérification ne porte donc pas sur une phrase du carnet, elle porte sur le code qui produit toutes les mesures des sections suivantes.

## 3. Itérer une tresse : la croissance d'une longueur

On part d'un élément du groupe libre (ici le générateur `x_1`) et on lui applique le **même** automorphisme `k` fois de suite. À chaque étape on mesure la longueur du mot réduit obtenu, puis le rapport entre deux longueurs consécutives.

Si ce rapport se stabilise, sa limite est le **taux de croissance** de l'itération : c'est la quantité dynamique que ce carnet cherche à mesurer.

In [4]:
def croissance(n, tresse, depart=(1,), etapes=14, plafond=3_000_000):
    '''Itere l'automorphisme de la tresse et rend les longueurs successives.'''
    automatismes = [sigma(n, abs(g), inverse=(g < 0)) for g in tresse]
    mot = reduire(depart)
    longueurs = []
    for _ in range(etapes):
        mot = appliquer(automatismes, mot)
        longueurs.append(len(mot))
        if len(mot) > plafond:          # garde memoire : la croissance est exponentielle
            break
    return longueurs


beta = [1, -2]          # sigma1 sigma2^-1 dans B_3
n = 3
longueurs = croissance(n, beta)
print("beta =", beta, " dans B_{}".format(n))
print()
print(" k | longueur | rapport au precedent")
print("---+----------+---------------------")
for k, L in enumerate(longueurs, start=1):
    if k == 1:
        print("{:2d} | {:8d} | {}".format(k, L, "-"))
    else:
        print("{:2d} | {:8d} | {:.6f}".format(k, L, L / longueurs[k - 2]))

dernier = longueurs[-1] / longueurs[-2]
phi_carre = (3 + math.sqrt(5)) / 2
print()
print("dernier rapport mesure : {:.6f}".format(dernier))
print("phi^2                  : {:.6f}".format(phi_carre))
print("ecart                  : {:.2e}".format(abs(dernier - phi_carre)))

beta = [1, -2]  dans B_3

 k | longueur | rapport au precedent
---+----------+---------------------
 1 |        3 | -
 2 |        9 | 3.000000
 3 |       25 | 2.777778
 4 |       67 | 2.680000
 5 |      177 | 2.641791
 6 |      465 | 2.627119
 7 |     1219 | 2.621505
 8 |     3193 | 2.619360
 9 |     8361 | 2.618541
10 |    21891 | 2.618227
11 |    57313 | 2.618108
12 |   150049 | 2.618062
13 |   392835 | 2.618045
14 |  1028457 | 2.618038

dernier rapport mesure : 2.618038
phi^2                  : 2.618034
ecart                  : 4.12e-06


### Lecture du résultat

Les longueurs croissent **exponentiellement** : le rapport entre deux itérations consécutives décroît régulièrement — `3.000000`, puis `2.777778`, `2.680000`, … — et se stabilise à `2.618038` sur la dernière itération. Le nombre d'or au carré vaut `2.618034` : l'écart affiché est de `4.12e-06`, et il continue de se réduire à chaque pas supplémentaire.

Deux précisions que la table permet de lire et que la prose ne doit pas escamoter :

- la convergence est **monotone par au-dessus** : chaque rapport dépasse `φ²` et s'en approche par valeurs décroissantes. Un rapport fini n'est donc pas le taux — c'est une **estimation** qui tend vers lui ;
- la longueur de départ (`x_1`, trois lettres après une itération) n'est pas une valeur propre : les premiers rapports sont donc les plus éloignés, et c'est l'asymptote qui porte l'information, pas le premier point.

Ce que cela dit : la tresse `σ1 σ2^{-1}` n'est ni une rotation ni un glissement monotone — elle **étire** les mots, et le taux d'étirement n'est pas un accident du mot de départ. Ce que cela ne dit pas encore : d'où sort ce nombre, et s'il se lit ailleurs que dans cette itération. C'est l'objet de la section suivante.

## 4. Le spectre de Perron : une borne, et l'endroit où elle est atteinte

Un automorphisme du groupe libre est décrit par les **mots images** de chaque générateur. On en extrait une matrice de transition `T` : `T[i][j]` compte combien de fois la lettre `x_j` (ou son inverse) apparaît dans l'image de `x_i`.

Comme l'image de `x_i` a une longueur majorée par la ligne `i` de `T`, la longueur de `β^k(x_i)` est majorée par une constante fois la somme de la ligne `i` de `T^k`. Le **rayon spectral** de `T` majore donc le taux de croissance **asymptotique** — la borne porte sur la limite des taux, pas sur un rapport fini, qui peut la dépasser d'un cheveu avant convergence. C'est un argument élémentaire, et il se vérifie sur les exemples.

La question empirique est ailleurs : **où la borne est-elle atteinte, et où ne l'est-elle pas ?**

In [5]:
def matrice_transition(n, tresse):
    T = np.zeros((n, n), dtype=int)
    automatismes = [sigma(n, abs(g), inverse=(g < 0)) for g in tresse]
    for i in range(1, n + 1):
        for g in appliquer(automatismes, (i,)):
            T[i - 1][abs(g) - 1] += 1
    return T


def rayon_spectral(M):
    valeurs = np.linalg.eigvals(M.astype(float))
    return float(max(abs(valeurs)))


cas = [
    ("B_3  sigma1 sigma2^-1        ", 3, [1, -2]),
    ("B_3  (sigma1 sigma2^-1)^2    ", 3, [1, -2, 1, -2]),
    ("B_3  sigma1 sigma2           ", 3, [1, 2]),
    ("B_3  Delta = sigma1 sigma2 sigma1", 3, [1, 2, 1]),
    ("B_4  sigma1 sigma3           ", 4, [1, 3]),
    ("B_4  sigma1 sigma2^-1 sigma3 sigma2^-1", 4, [1, -2, 3, -2]),
]

print("cas                                       | long. finale | rapport   | rho(T)    | rho - rapport")
print("------------------------------------------+--------------+-----------+-----------+--------------")
for nom, n, tresse in cas:
    L = croissance(n, tresse, etapes=12)
    rapport = L[-1] / L[-2]
    rho = rayon_spectral(matrice_transition(n, tresse))
    print("{} | {:12d} | {:.6f}  | {:.6f}  | {:+.6f}".format(nom, L[-1], rapport, rho, rho - rapport))

cas                                       | long. finale | rapport   | rho(T)    | rho - rapport
------------------------------------------+--------------+-----------+-----------+--------------
B_3  sigma1 sigma2^-1         |       150049 | 2.618062  | 2.618034  | -0.000028


B_3  (sigma1 sigma2^-1)^2     |      7049155 | 6.854108  | 6.854102  | -0.000006
B_3  sigma1 sigma2            |           25 | 1.000000  | 2.831177  | +1.831177
B_3  Delta = sigma1 sigma2 sigma1 |           37 | 1.057143  | 3.732051  | +2.674908
B_4  sigma1 sigma3            |           25 | 1.086957  | 2.414214  | +1.327257


B_4  sigma1 sigma2^-1 sigma3 sigma2^-1 |      7381965 | 6.854135  | 7.910856  | +1.056721


### Lecture du résultat

La table sépare deux régimes, et c'est le résultat de la section :

- **la borne est atteinte, à la convergence près** pour les deux tresses de `B_3` qui étirent réellement — `σ1σ2^{-1}` et son carré : l'écart `ρ(T) − rapport` vaut `-0.000028` et `-0.000006`, soit un résidu de convergence et non une violation. Ces deux valeurs sont **négatives** parce que le rapport décroît vers sa limite par au-dessus (section 3) : la borne porte sur la **limite des taux**, pas sur chaque rapport fini, et l'itération finie peut donc la dépasser d'un cheveu ;
- **la borne n'est pas atteinte du tout** pour `σ1σ2`, pour `Δ = σ1σ2σ1` et pour `σ1σ3` — des tresses **réductibles ou périodiques** : `σ1σ2` stabilise même la longueur (`rapport 1.000000`), à un écart de `+1.83` de sa borne ;
- et en `B_4`, la tresse `σ1 σ2^{-1} σ3 σ2^{-1}` étire pourtant vite (`6.854135`) tout en restant **loin** de son rayon spectral (`7.910856`) : la borne élémentaire n'est donc **pas serrée en général**, elle majore sans décrire.

Autrement dit : `ρ(T)` est une borne calculable **gratuitement**, qui sépare les cas qui étirent de ceux qui n'étirent pas, mais elle ne **remplace pas** la dilatation — dans `B_4` on mesure un écart de plus d'une unité entre croissance et borne. La vraie dilatation se calcule par les trains tracks de Bestvina–Handel, que ce carnet ne met pas en œuvre ; la section suivante nomme précisément ce que ce détour laisse sur la table.

## 5. Le niveau abélianisé, qui ne voit rien

Il reste à expliquer pourquoi il fallait passer par le **groupe libre**. L'observateur naturel d'une tresse est son effet sur l'**abélianisé** `Z^n` — la somme des exposants de chaque générateur, c'est-à-dire ce que voit le premier groupe d'homologie.

Le calcul se fait en somme d'exposants : à chaque lettre rencontrée on ajoute `+1` ou `-1` dans la ligne de son générateur. La matrice obtenue est entière ; son spectre dit tout ce que le niveau abélianisé est capable de dire.

In [6]:
def matrice_abelianisee(n, tresse):
    A = np.zeros((n, n), dtype=int)
    automatismes = [sigma(n, abs(g), inverse=(g < 0)) for g in tresse]
    for i in range(1, n + 1):
        for g in appliquer(automatismes, (i,)):
            A[abs(g) - 1][i - 1] += 1 if g > 0 else -1
    return A


print("cas                                       | A (colonnes = images)                  | rho(A)")
print("------------------------------------------+----------------------------------------+-------")
for nom, n, tresse in cas:
    A = matrice_abelianisee(n, tresse)
    lignes = " ".join(str(ligne.tolist()) for ligne in A)
    print("{} | {:<38} | {:.6f}".format(nom, lignes, rayon_spectral(A)))

cas                                       | A (colonnes = images)                  | rho(A)
------------------------------------------+----------------------------------------+-------
B_3  sigma1 sigma2^-1         | [0, 1, 0] [0, 0, 1] [1, 0, 0]          | 1.000000
B_3  (sigma1 sigma2^-1)^2     | [0, 0, 1] [1, 0, 0] [0, 1, 0]          | 1.000000
B_3  sigma1 sigma2            | [0, 1, 0] [0, 0, 1] [1, 0, 0]          | 1.000000
B_3  Delta = sigma1 sigma2 sigma1 | [0, 0, 1] [0, 1, 0] [1, 0, 0]          | 1.000000
B_4  sigma1 sigma3            | [0, 1, 0, 0] [1, 0, 0, 0] [0, 0, 0, 1] [0, 0, 1, 0] | 1.000000
B_4  sigma1 sigma2^-1 sigma3 sigma2^-1 | [0, 1, 0, 0] [0, 0, 0, 1] [0, 0, 1, 0] [1, 0, 0, 0] | 1.000000


### Lecture du résultat

Toutes les matrices abélianisées sont des **matrices de permutation**, et leur rayon spectral vaut `1` dans chaque cas — y compris pour la tresse dont la croissance mesurée vaut `φ² ≈ 2,618`.

Ce n'est pas une anomalie de calcul, c'est le fait central : au niveau de l'homologie, une tresse n'est qu'un réarrangement de brins, et un réarrangement ne dilate rien. Toute la dynamique de la section 3 vit dans ce que l'abélianisation **jette** : l'ordre des lettres, donc les conjugaisons `x_i · x_{i+1} · x_i^{-1}`, qui sont invisibles à la somme des exposants.

C'est la raison d'être du détour par le groupe libre : un invariant qui ne distingue pas une dilatation d'une permutation ne peut pas répondre à la question du carnet.

## Exercices

Trois exercices, chacun vérifiable par une assertion que tu écris toi-même. Les cellules s'exécutent même sans être complétées : elles affichent ce qu'il reste à faire.

### Exercice 1 — La puissance double l'exposant

La section 3 mesure un taux de `φ²` pour `σ1 σ2^{-1}` ; la section 4 mesure `φ⁴` pour son carré. La loi apparente est donc `φ^{2k}` pour la puissance `k`.

**À faire** : vérifie cette loi **par la mesure** et non par la formule — calcule le taux de `(σ1 σ2^{-1})^k` pour `k = 1, 2, 3`, compare chacun à `φ^{2k}`, et explique le facteur 2 (indice : le rayon spectral d'une puissance de matrice est la puissance du rayon spectral, et la matrice de transition de `β^k` n'est pas `T` mais son itérée).

In [7]:
def taux(n, tresse, etapes=12):
    L = croissance(n, tresse, etapes=etapes)
    return L[-1] / L[-2]


# TODO etudiant : construire (sigma1 sigma2^-1)^k pour k = 1, 2, 3,
# mesurer le taux de chacune, et le confronter a phi**(2*k).
phi = (1 + math.sqrt(5)) / 2
for k in (1, 2, 3):
    print("k = {} : phi**(2k) = {:.6f} -- taux mesure a completer".format(k, phi ** (2 * k)))
print("Exercice a completer : la loi se verifie sur trois mesures, pas sur une extrapolation.")

k = 1 : phi**(2k) = 2.618034 -- taux mesure a completer
k = 2 : phi**(2k) = 6.854102 -- taux mesure a completer
k = 3 : phi**(2k) = 17.944272 -- taux mesure a completer
Exercice a completer : la loi se verifie sur trois mesures, pas sur une extrapolation.


### Exercice 2 — Un témoin qui ne croît pas

Toutes les tresses ne dilatent pas. La section 4 en montre trois, mais sans les caractériser autrement que par leur taux mesuré.

**À faire** : construis une tresse de `B_3` dont le taux mesuré reste **proche de 1** sur au moins douze itérations, et affiche la suite des rapports pour le prouver — une valeur finale seule ne suffit pas à distinguer un plateau d'une convergence lente.

In [8]:
# TODO etudiant : proposer un mot de B_3, mesurer ses rapports successifs,
# et n'accepter le resultat que si la suite entiere reste proche de 1.
def rapports(n, tresse, etapes=12):
    L = croissance(n, tresse, etapes=etapes)
    return [L[i] / L[i - 1] for i in range(1, len(L))]


print("Exercice a completer : afficher la suite complete des rapports, pas seulement le dernier.")

Exercice a completer : afficher la suite complete des rapports, pas seulement le dernier.


### Exercice 3 — La borne tient-elle quand on change le mot de départ ?

Les mesures des sections 3 et 4 partent du générateur `x_1`. Rien ne garantit *a priori* que le taux mesuré soit indépendant du point de départ.

**À faire** : reprends `σ1 σ2^{-1}` et mesure le taux en partant de `x_1`, de `x_2` et d'un mot composé ; compare avec `ρ(T)`. Si les trois taux diffèrent, dis-le : c'est le signe que la mesure dépend de l'orbite choisie, et la phrase du carnet qui parle de « taux de l'itération » demande alors une précision.

In [9]:
# TODO etudiant : comparer les taux pour plusieurs mots de depart,
# puis les confronter au rayon spectral de la matrice de transition.
departs = [(1,), (2,), (1, 2, 1)]
print("Exercice a completer : trois departs, un seul rho(T) -- qui a raison ?")
for d in departs:
    print("  depart", d, "-> a mesurer")

Exercice a completer : trois departs, un seul rho(T) -- qui a raison ?
  depart (1,) -> a mesurer
  depart (2,) -> a mesurer
  depart (1, 2, 1) -> a mesurer


## Résumé

Ce carnet a rendu **mesurable** la dynamique d'une tresse, en trois actes qui se contrôlent l'un l'autre.

1. **Relations d'Artin vérifiées** : l'inverse, la relation de tresse et la commutation des générateurs distants tiennent comme égalités de mots du groupe libre pour `n` de 2 à 5. La vérification porte sur le code qui produit toutes les mesures, pas sur une phrase.
2. **Croissance mesurée** : l'itération de `σ1 σ2^{-1}` dans `B_3` étire la longueur réduite à un taux qui se stabilise sur `φ²` ; le rapport entre deux itérations consécutives s'y ancre en quelques pas, et l'écart au nombre d'or au carré est affiché, pas arrondi en prose.
3. **Spectre de Perron situé** : `ρ(T)` sépare nettement les tresses qui étirent des tresses réductibles ou périodiques, atteint exactement la croissance mesurée dans les cas pseudo-Anosov de `B_3`, et **ne l'atteint pas** en `B_4` — la borne est une borne, pas une identité.
4. **Le niveau abélianisé ne voit rien** : toutes les matrices d'homologie du corpus sont des permutations de rayon spectral `1`, y compris pour la tresse la plus dilatante. L'ordre des lettres est l'information qui manque à l'homologie, et c'est lui que le groupe libre conserve.

### Ce que le carnet laisse ouvert

La **dilatation** au sens de Nielsen–Thurston se calcule sur un train track (Bestvina–Handel) : elle n'est pas obtenue ici, et aucun paragraphe ne la simule. Le taux mesuré, la borne `ρ(T)` et la dilatation sont trois objets distincts ; ce carnet mesure les deux premiers et nomme le troisième.

### Pour aller plus loin

- KNOTS-02 et KNOTS-04 pour les invariants et la géométrie du nœud fermé de ces tresses ;
- les exercices ci-dessus, chacun avec son assertion à écrire ;
- `docs/` de la série pour la place de ce carnet dans le parcours.
***
**Navigation** : [<< Lean-17d — Géométrie hyperbolique](KNOTS-04-Geometrie-Hyperbolique-Python.ipynb) | [Index](README.md)